In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report
)

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

In [ ]:
data = pd.read_csv("https://raw.githubusercontent.com/brina948/Deep-Learning/refs/heads/main/datasets/cross-sectional/bank-additional%5B1%5D.csv",sep=";")
data.head()

# **Data Pre-Processing**

**Informasi Data**

In [ ]:
data.info()
display(data.describe(include="all").T)

**Missing Value**

In [ ]:
missing_count = data.isna().sum()
missing_percent = (data.isna().mean() * 100).round(2)

missing_count

**Duplikat**

In [ ]:
print("Jumlah baris duplikat:", data.duplicated().sum())

**Imbalance Data**

In [ ]:
sns.countplot(data=data, x="y")
plt.title("Distribusi Target: Subscription")
plt.xlabel("Subscribed?")
plt.ylabel("Number of Records")
plt.show()

**Spli Training dan Testing**

In [ ]:
data_model = data.drop(columns=["duration"],errors="ignore").copy()

X = data_model.drop(columns=["y"])
y = data_model["y"].map({"no": 0,"yes": 1})

In [ ]:
# Tahap 1: 60% training dan 40% data sementara
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40,random_state=42, stratify=y)

# Tahap 2: data sementara dibagi sama rata
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42,stratify=y_temp)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

**Pisahkan fitur numerik dan kategorikal**

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Fitur numerik:", numeric_features)
print("Fitur kategorikal:", categorical_features)

**Tangani missing value numerik**

In [ ]:
# Hitung median dari training saja
median_values = X_train[numeric_features].median()

# Isi missing value
X_train[numeric_features] = (X_train[numeric_features].fillna(median_values))
X_val[numeric_features] = (X_val[numeric_features].fillna(median_values))
X_test[numeric_features] = (X_test[numeric_features].fillna(median_values))

**Standardisasi fitur numerik**

In [ ]:
scaler = StandardScaler()

# Fit dan transform hanya pada training
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])

# Validation dan testing hanya ditransformasi
X_val[numeric_features] = scaler.transform(X_val[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

**Tangani missing value kategorikal**

In [ ]:
# Hitung modus dari training saja
mode_values = X_train[categorical_features].mode().iloc[0]

X_train[categorical_features] = (X_train[categorical_features].fillna(mode_values))
X_val[categorical_features] = (X_val[categorical_features].fillna(mode_values))
X_test[categorical_features] = (X_test[categorical_features].fillna(mode_values))

**One-Hot Encoding**

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore",sparse_output=False)

# Fit dan transform training
X_train_cat = encoder.fit_transform(X_train[categorical_features])

# Transform validation dan testing
X_val_cat = encoder.transform(X_val[categorical_features])
X_test_cat = encoder.transform(X_test[categorical_features])

**Gabungkan fitur numerik dan kategorikal**

In [ ]:
# Ambil fitur numerik
X_train_num = X_train[numeric_features].to_numpy()
X_val_num = X_val[numeric_features].to_numpy()
X_test_num = X_test[numeric_features].to_numpy()

# Gabungkan fitur numerik dan kategorikal
X_train_processed = np.concatenate([X_train_num, X_train_cat], axis=1).astype("float32")
X_val_processed = np.concatenate([X_val_num, X_val_cat], axis=1).astype("float32")
X_test_processed = np.concatenate([X_test_num, X_test_cat], axis=1).astype("float32")

# Target
y_train_array = y_train.to_numpy(dtype="float32")
y_val_array = y_val.to_numpy(dtype="float32")
y_test_array = y_test.to_numpy(dtype="float32")

# Jumlah node input
n_features = X_train_processed.shape[1]

print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Testing:", X_test_processed.shape)
print("Jumlah node input:", n_features)

# **Modelling**

**Membangun model baseline**

In [ ]:
baseline_model = Sequential([
    Input(shape=(n_features,)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

# optimizer=tf.keras.optimizers.Adam(learning_rate=0.01)
# optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.01)
baseline_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

baseline_model.summary()

**Training model baseline**

In [ ]:
history = baseline_model.fit(
    X_train_processed,
    y_train_array,
    validation_data=(
        X_val_processed,
        y_val_array
    ),
    epochs=50,
    batch_size=32,
    verbose=1
)

# **Evaluasi**

**Visualisasi training dan validation loss**

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

**Evaluation pada data testing**

In [ ]:
# Prediksi probabilitas kelas positif
y_test_prob = baseline_model.predict(X_test_processed, verbose=0).ravel()

# Konversi probabilitas menjadi label kelas
y_test_pred = (y_test_prob >= 0.5).astype(int)

# Membuat confusion matrix
cm = confusion_matrix(y_test_array, y_test_pred)

# Visualisasi heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["no", "yes"], yticklabels=["no", "yes"])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Baseline Model")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(y_test_array.astype(int), y_test_pred, labels=[0, 1],
                            target_names=["No", "Yes"], digits=4, zero_division=0))